# 04 — Model Explainability (SHAP)

Uses `src/models/explain.py` to compute SHAP values for the best classifier.

**Key insight:** After removing PRICE_PER_SQFT (data leakage), the top predictors are:
1. DIST_MANHATTAN_CENTER (0.212) — location is king
2. PROPERTYSQFT (0.184) — size matters
3. BATH (0.166) — luxury indicator

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import joblib
import pandas as pd
import numpy as np
import plotly.express as px

from src.config import MODELS_DIR, PRICE_ZONE_LABELS, RANDOM_SEED, TEST_SIZE
from src.models.explain import compute_shap_values, global_feature_importance, get_top_features_for_prediction
from run_training import prepare_data, get_feature_df
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [ ]:
# Load model + prepare test data
df, y_zone, y_log_price, borough = prepare_data()
X = get_feature_df(df)
le = LabelEncoder()
y_enc = le.fit_transform(y_zone)
_, X_test, _, yz_test, _, _, _, _ = train_test_split(
    X, y_enc, y_log_price, borough, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y_enc)

clf = joblib.load(MODELS_DIR / "price_zone_best.joblib")
preprocessor = clf.named_steps["preprocessor"]
classifier = clf.named_steps["classifier"]

X_test_t = preprocessor.transform(X_test)
feature_names = list(preprocessor.get_feature_names_out())
print(f"Transformed features: {len(feature_names)}")

## Global Feature Importance (SHAP)

> Note: SHAP computation may take several minutes for 200 samples.

In [ ]:
X_test_df = pd.DataFrame(X_test_t, columns=feature_names)
shap_values, explainer = compute_shap_values(classifier, X_test_df, max_samples=100)
importance_df = global_feature_importance(shap_values, feature_names)

# Plot top 15 features
top15 = importance_df.head(15)
fig = px.bar(top15, x="mean_abs_shap", y="feature", orientation="h",
             title="Top 15 Features by Mean |SHAP| Value",
             labels={"mean_abs_shap": "Mean |SHAP|", "feature": ""},
             color="mean_abs_shap", color_continuous_scale="Blues")
fig.update_layout(yaxis=dict(autorange="reversed"), height=500)
fig.show()

## Per-Prediction Explanation (sample)

In [ ]:
# Explain a single prediction
idx = 0
top_factors = get_top_features_for_prediction(explainer, shap_values, feature_names, idx=idx, top_n=5)
print(f"Top 5 factors for sample {idx}:")
for factor in top_factors:
    print(f"  {factor['direction']} {factor['feature']:30s} SHAP={factor['shap_value']:+.4f}")